In [3]:
from google.colab import files
uploaded = files.upload()


Saving sentiment-analysis.csv to sentiment-analysis.csv


In [4]:
# Inspect first few lines of the file to see what's happening
from pathlib import Path

path = Path('sentiment-analysis.csv')
with path.open('r', encoding='utf-8', errors='replace') as f:
    for i, line in enumerate(f):
        print(f"Line {i+1} repr: {repr(line.strip())}")
        if i >= 4:
            break


Line 1 repr: '"Text, Sentiment, Source, Date/Time, User ID, Location, Confidence Score"'
Line 2 repr: '"""I love this product!"", Positive, Twitter, 2023-06-15 09:23:14, @user123, New York, 0.85"'
Line 3 repr: '"""The service was terrible."", Negative, Yelp Reviews, 2023-06-15 11:45:32, user456, Los Angeles, 0.65"'
Line 4 repr: '"""This movie is amazing!"", Positive, IMDb, 2023-06-15 14:10:22, moviefan789, London, 0.92"'
Line 5 repr: '"""I\'m so disappointed with their customer support."", Negative, Online Forum, 2023-06-15 17:35:11, forumuser1, Toronto, 0.78"'


In [5]:
from pathlib import Path

p = Path('sentiment-analysis.csv')
lines = p.read_text(encoding='utf-8', errors='replace').splitlines()

cleaned_lines = []
for line in lines:
    # Remove the outer quotes if they exist
    if line.startswith('"') and line.endswith('"'):
        line = line[1:-1]
    # Replace doubled quotes "" with single "
    line = line.replace('""', '"')
    cleaned_lines.append(line)

# Write the cleaned data to a new file
clean_path = Path('sentiment-analysis-clean.csv')
clean_path.write_text("\n".join(cleaned_lines), encoding='utf-8')

print(" Cleaned file saved as:", clean_path)


 Cleaned file saved as: sentiment-analysis-clean.csv


In [6]:
import pandas as pd
# After uploading, read it into a DataFrame
df = pd.read_csv('sentiment-analysis-clean.csv', quotechar='"', skipinitialspace=True)
# Check a few rows
df.head()

,Text,Sentiment,Source,Date/Time,User ID,Location,Confidence Score
0,I love this product!,Positive,Twitter,2023-06-15 09:23:14,@user123,New York,0.85
1,The service was terrible.,Negative,Yelp Reviews,2023-06-15 11:45:32,user456,Los Angeles,0.65
2,This movie is amazing!,Positive,IMDb,2023-06-15 14:10:22,moviefan789,London,0.92
3,I'm so disappointed with their customer support.,Negative,Online Forum,2023-06-15 17:35:11,forumuser1,Toronto,0.78
4,Just had the best meal of my life!,Positive,TripAdvisor,2023-06-16 08:50:59,foodie22,Paris,0.88


In [7]:
print(df.columns)


Index(['Text', 'Sentiment', 'Source', 'Date/Time', 'User ID', 'Location',
       'Confidence Score'],
      dtype='object')


In [8]:
df['Sentiment'].value_counts()


,count
Sentiment,
Positive,53
Negative,43


!pip install transformers datasets torch scikit-learn

In [9]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [10]:
label_map = {'Negative': 0, 'Positive': 1}

df['Sentiment'] = df['Sentiment'].str.strip().str.capitalize()
df['label'] = df['Sentiment'].map(label_map)

df[['Sentiment', 'label']].drop_duplicates()


,Sentiment,label
0,Positive,1
1,Negative,0


In [29]:
from sklearn.model_selection import train_test_split
import pandas as pd
from datasets import Dataset

#  Step 1: Remove duplicates (important!)
df = df.drop_duplicates(subset=['Text']).reset_index(drop=True)

#  Step 2: Split into train and validation sets (stratified)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['Text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']  # keeps class balance same
)

#  Step 3: Check for accidental overlap (should be 0)
overlap = set(train_texts).intersection(set(val_texts))
print(f"Overlap count: {len(overlap)}")
assert len(overlap) == 0, " Train and validation sets have overlapping texts!"

#  Step 4: Create DataFrames
train_df = pd.DataFrame({'text': train_texts, 'labels': train_labels})
val_df = pd.DataFrame({'text': val_texts, 'labels': val_labels})

#  Step 5: Convert to HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

print(" Train and validation datasets created successfully!")
print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")


Overlap count: 0
 Train and validation datasets created successfully!
Train samples: 60
Validation samples: 15


In [30]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

In [31]:
import os
os.environ["WANDB_DISABLED"] = "true"


In [32]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 1️⃣ Model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# 2️⃣ Training arguments (compatible with older transformers)
training_args = TrainingArguments(
    output_dir='./results',
    do_eval=True,                    # enable evaluation
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

# 3 Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {"accuracy": acc, "f1": f1}

# 4 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics  # metrics during evaluation
)

# 5 Train
trainer.train()

# 6️⃣ Save the fine-tuned model and tokenizer
trainer.save_model("./my_finetuned_model")          # Saves model weights and config
tokenizer.save_pretrained("./my_finetuned_model")   # Saves tokenizer files


# 6 Evaluate manually (optional)
eval_results = trainer.evaluate()
print(eval_results)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-437323716.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10,0.625000
20,0.466100


{'eval_loss': 0.4229026436805725, 'eval_accuracy': 0.9333333333333333, 'eval_f1': 0.9333333333333333, 'eval_runtime': 0.1614, 'eval_samples_per_second': 92.936, 'eval_steps_per_second': 12.392, 'epoch': 3.0}


In [33]:
#Evaluation Metrics (Accuracy, F1, Confusion Matrix)
from sklearn.metrics import confusion_matrix, classification_report

# Make predictions
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=-1)

# True labels
val_labels = [item['labels'].item() for item in val_dataset]

# Accuracy and F1
from sklearn.metrics import accuracy_score, f1_score
print("Accuracy:", accuracy_score(val_labels, preds))
print("F1-score:", f1_score(val_labels, preds, average='weighted'))

# Confusion Matrix
print("\nConfusion Matrix:\n", confusion_matrix(val_labels, preds))

# Classification Report
print("\nClassification Report:\n", classification_report(val_labels, preds))


Accuracy: 0.9333333333333333
F1-score: 0.9333333333333333

Confusion Matrix:
 [[7 0]
 [1 7]]

Classification Report:
               precision    recall  f1-score   support

           0       0.88      1.00      0.93         7
           1       1.00      0.88      0.93         8

    accuracy                           0.93        15
   macro avg       0.94      0.94      0.93        15
weighted avg       0.94      0.93      0.93        15



In [34]:
#Example Predictions (Inference on New Feedback)
import torch

# Load fine-tuned model and tokenizer
model = BertForSequenceClassification.from_pretrained("./my_finetuned_model")
tokenizer = BertTokenizer.from_pretrained("./my_finetuned_model")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

label_map = {"positive": 1, "negative": 0}  # adjust if neutral
inv_label_map = {v: k for k, v in label_map.items()}

sample_texts = [
    "The product quality is amazing!",
    "I am disappointed with the service.",
    "It's okay, not great but not bad."
]

encoded = tokenizer(sample_texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
encoded = {k: v.to(device) for k, v in encoded.items()}

with torch.no_grad():
    outputs = model(**encoded)
    predicted = outputs.logits.argmax(dim=1).tolist()

for text, label in zip(sample_texts, predicted):
    print(f"Text: {text}\nPredicted Sentiment: {inv_label_map[label]}\n")


Text: The product quality is amazing!
Predicted Sentiment: positive

Text: I am disappointed with the service.
Predicted Sentiment: negative

Text: It's okay, not great but not bad.
Predicted Sentiment: negative



In [17]:
!pip install gradio


In [35]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import gradio as gr

# 1️⃣ Load your fine-tuned model and tokenizer
model_path = "./my_finetuned_model"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)

# 2️⃣ Move model to GPU (if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 3️⃣ Label mapping — must match your training labels
label_map = {0: "negative", 1: "positive"}

# 4️⃣ Define the prediction function
def predict_sentiment(text):
    encoded = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item()

    sentiment = label_map[pred]
    return {sentiment: confidence}

# 5️⃣ Build Gradio interface
iface = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=3, placeholder="Type customer feedback here..."),
    outputs=gr.Label(num_top_classes=2),
    title="Customer Feedback Sentiment Classifier (BERT)",
    description="Enter feedback text and get the predicted sentiment (Positive / Negative)."
)

# 6️⃣ Launch interface (works in Colab)
iface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://35bdaebe03d4621f60.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [27]:
set(train_texts).intersection(set(val_texts))


{"I can't stop listening to this song. It's incredible!",
 "I can't stop listening to this song. It's my new favorite!",
 "Just had the most amazing vacation! I can't wait to go back.",
 'The quality of this product is subpar.',
 'The website loading speed is frustratingly slow. Needs improvement.',
 'This movie is amazing!'}